In [1]:
import sqlalchemy
import psycopg

print("SQLAlchemy:", sqlalchemy.__version__)
print("Psycopg:", psycopg.__version__)

SQLAlchemy: 2.0.51
Psycopg: 3.3.4


In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()  # reads PGPASSWORD from the project's .env file

connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=os.getenv("PGPASSWORD"),
    host="localhost",
    port=5432,
    database="olist_analytics",
)

engine = create_engine(connection_url)

In [3]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.scalar())

1


In [4]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user")
    )
    
    print(result.fetchone())

('olist_analytics', 'postgres')


In [5]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT version()")
    )
    
    print(result.scalar())

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit


In [23]:
import pandas as pd
import psycopg
import os
from pathlib import Path
from dotenv import load_dotenv

In [33]:
load_dotenv()

# PROJECT_ROOT = Path.cwd().parent
# PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

PROCESSED_DIR = Path("../dataset/processed")

# print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
print("Processed directory exists:", PROCESSED_DIR.exists())

PG_PARAMS = dict(
    host="localhost", port=5432, dbname="olist_analytics",
    user="postgres", password=os.getenv("PGPASSWORD"),
)

Processed directory: ..\dataset\processed
Processed directory exists: True


In [36]:
# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------
def to_nullable_int(df, cols):
    """Float64-with-NaN (or object-with-pd.NA) -> pandas nullable Int64.
    Fixes the '225.0' vs '225' COPY error for any INT/SMALLINT target column
    that can legitimately be NULL in the source."""
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("Int64")
    return df

def to_nullable_bool(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("boolean")
    return df

def zpad_zip(series, width=5):
    """Guards against int64-stored zip prefixes silently losing leading zeros."""
    def fmt(z):
        if pd.isna(z):
            return None
        s = str(int(z)) if isinstance(z, float) else str(z).strip()
        return s.zfill(width)
    return series.apply(fmt)

def df_to_rows(df, cols):
    sub = df[cols].astype(object).where(pd.notna(df[cols]), None)
    return list(sub.itertuples(index=False, name=None))

def bulk_copy(conn, table, cols, rows):
    col_list = ", ".join(cols)
    with conn.cursor() as cur:
        with cur.copy(f"COPY {table} ({col_list}) FROM STDIN") as copy:
            for row in rows:
                copy.write_row(row)
    print(f"  loaded {len(rows):>7,} rows -> {table}")

def to_date_key(ts):
    return None if pd.isna(ts) else int(pd.Timestamp(ts).strftime("%Y%m%d"))

def fetch_key_map(conn, table, key_col, natural_col):
    with conn.cursor() as cur:
        cur.execute(f"SELECT {natural_col}, {key_col} FROM {table}")
        return dict(cur.fetchall())

def fk_report_and_validate(df, fk_specs, table_name):
    print(f"\n--- {table_name}: FK validation ---")
    print(f"  source rows: {len(df):,}")
    problems = {}
    for key_col, natural_col in fk_specs:
        unresolved_mask = df[key_col].isna()
        n = int(unresolved_mask.sum())
        print(f"  unresolved {key_col:<22} (via {natural_col}): {n}")
        if n > 0:
            problems[key_col] = df.loc[unresolved_mask, natural_col].unique()[:15]
    rows_to_load = len(df) if not problems else 0
    rows_rejected = len(df) if problems else 0
    print(f"  rows to load: {rows_to_load:,}   rows rejected: {rows_rejected:,}")
    if problems:
        lines = [f"ETL STOPPED — {table_name}: unresolved foreign keys (zero rows loaded for this table):"]
        for key_col, sample in problems.items():
            lines.append(f"    {key_col}: sample unresolved natural keys -> {list(sample)}")
        raise RuntimeError("\n".join(lines))
    print(f"  \u2713 all foreign keys resolved")

def print_final_summary(conn, tables):
    print("\n=== FINAL LOAD SUMMARY (post-commit) ===")
    with conn.cursor() as cur:
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            print(f"  {t:<20} {cur.fetchone()[0]:>8,} rows")


conn = psycopg.connect(**PG_PARAMS)
conn.autocommit = False


In [37]:
try:
    # 1) DIM_GEOLOCATION
    dim_geo = pd.read_parquet(PROCESSED_DIR / "dim_geo.parquet").rename(
        columns={"geolocation_zip_code_prefix": "zip_code_prefix"}
    )
    dim_geo["zip_code_prefix"] = zpad_zip(dim_geo["zip_code_prefix"])
    dim_geo = to_nullable_int(dim_geo, ["n_points_used"])
    cols = ["zip_code_prefix", "lat", "lon", "n_points_used", "span_km", "confidence"]
    bulk_copy(conn, "dim_geolocation", cols, df_to_rows(dim_geo, cols))

    # 2) DIM_CUSTOMERS
    customers = pd.read_parquet(PROCESSED_DIR / "customers.parquet")
    order_master = pd.read_parquet(PROCESSED_DIR / "order_master.parquet")

    cust_with_ts = customers.merge(
        order_master[["customer_id", "order_purchase_timestamp"]], on="customer_id", how="left"
    )
    dim_customers_df = (
        cust_with_ts.sort_values(["customer_unique_id", "order_purchase_timestamp"])
        .drop_duplicates(subset="customer_unique_id", keep="last")
        .rename(columns={"customer_zip_code_prefix": "zip_code_prefix",
                          "customer_city": "city", "customer_state": "state"})
    )
    dim_customers_df["zip_code_prefix"] = zpad_zip(dim_customers_df["zip_code_prefix"])
    cols = ["customer_unique_id", "zip_code_prefix", "city", "state"]
    bulk_copy(conn, "dim_customers", cols, df_to_rows(dim_customers_df, cols))

    # 3) DIM_PRODUCTS
    products = pd.read_parquet(PROCESSED_DIR / "products_clean.parquet").rename(columns={
        "product_category_name": "category_name",
        "product_category_name_english": "category_english",
        "product_weight_g": "weight_g", "product_length_cm": "length_cm",
        "product_height_cm": "height_cm", "product_width_cm": "width_cm",
        "product_photos_qty": "photos_qty", "product_name_lenght": "name_length",
        "product_description_lenght": "description_length",
    })
    products = to_nullable_int(products, ["weight_g", "length_cm", "height_cm",
                                            "width_cm", "photos_qty", "name_length", "description_length"])
    cols = ["product_id", "category_name", "category_english", "weight_g",
            "length_cm", "height_cm", "width_cm", "photos_qty", "name_length", "description_length"]
    bulk_copy(conn, "dim_products", cols, df_to_rows(products, cols))

    # 4) DIM_SELLERS
    sellers = pd.read_parquet(PROCESSED_DIR / "sellers.parquet").rename(columns={
        "seller_zip_code_prefix": "zip_code_prefix", "seller_city": "city", "seller_state": "state",
    })
    sellers["zip_code_prefix"] = zpad_zip(sellers["zip_code_prefix"])
    cols = ["seller_id", "zip_code_prefix", "city", "state"]
    bulk_copy(conn, "dim_sellers", cols, df_to_rows(sellers, cols))

    print("\nDimensions loaded (uncommitted). Fetching surrogate keys within the same transaction...")
    customer_key_map = fetch_key_map(conn, "dim_customers", "customer_key", "customer_unique_id")
    product_key_map  = fetch_key_map(conn, "dim_products", "product_key", "product_id")
    seller_key_map   = fetch_key_map(conn, "dim_sellers", "seller_key", "seller_id")

    # 5) FACT_ORDERS
    assert "customer_unique_id" in order_master.columns, \
    "order_master is missing customer_unique_id — re-check the Week 1 export"
    om = order_master.copy()

    # ---------------------------------------------------------------
    # Genuine repeat customer flag
    # Week 1 definition:
    # repeat customer = >1 order AND elapsed time between
    # first and last purchase is > 0 whole days
    # ---------------------------------------------------------------
    cust_span = om.groupby("customer_unique_id")["order_purchase_timestamp"].agg(
        n_orders="count",
        first_purchase="min",
        last_purchase="max"
    )

    cust_span["days_between_first_last"] = (
        cust_span["last_purchase"] - cust_span["first_purchase"]
    ).dt.days

    cust_span["is_genuine_repeat_customer"] = (
        (cust_span["n_orders"] > 1) &
        (cust_span["days_between_first_last"] > 0)
    )

    om = om.merge(
        cust_span[["is_genuine_repeat_customer"]],
        on="customer_unique_id",
        how="left"
    )

    om["customer_key"] = om["customer_unique_id"].map(customer_key_map).astype("Int64")
    om["purchase_date_key"] = pd.array(om["order_purchase_timestamp"].apply(to_date_key), dtype="Int64")
    om["delivered_date_key"] = pd.array(om["order_delivered_customer_date"].apply(to_date_key), dtype="Int64")
    om["estimated_delivery_date_key"] = pd.array(om["order_estimated_delivery_date"].apply(to_date_key), dtype="Int64")

    delivered_mask = (om["order_status"] == "delivered") & (~om["delivery_date_missing_flag"])
    om["delivery_days"] = pd.NA
    om["days_late"] = pd.NA
    om["is_late"] = pd.NA
    om.loc[delivered_mask, "delivery_days"] = (
        om.loc[delivered_mask, "order_delivered_customer_date"] - om.loc[delivered_mask, "order_purchase_timestamp"]
    ).dt.days
    om.loc[delivered_mask, "days_late"] = (
        om.loc[delivered_mask, "order_delivered_customer_date"] - om.loc[delivered_mask, "order_estimated_delivery_date"]
    ).dt.days
    om.loc[delivered_mask, "is_late"] = om.loc[delivered_mask, "days_late"] > 0

    om = to_nullable_int(om, ["n_items", "n_payment_methods", "max_installments", "delivery_days", "days_late"])
    om = to_nullable_bool(om, ["is_late", "is_genuine_repeat_customer"])

    fk_report_and_validate(om, [("customer_key", "customer_unique_id")], "fact_orders")

    cols = ["order_id", "customer_key", "purchase_date_key", "delivered_date_key",
            "estimated_delivery_date_key", "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date",
            "order_estimated_delivery_date", "order_status", "n_items", "total_price",
            "total_freight", "total_payment_value", "n_payment_methods", "max_installments",
            "delivery_days", "days_late", "is_late", "delivery_date_missing_flag", "is_genuine_repeat_customer"]
    bulk_copy(conn, "fact_orders", cols, df_to_rows(om, cols))

    order_key_map = fetch_key_map(conn, "fact_orders", "order_key", "order_id")

    # 6) FACT_ORDER_ITEMS
    order_items = pd.read_parquet(PROCESSED_DIR / "order_items.parquet").rename(
        columns={"order_item_id": "order_item_seq"}
    )
    order_items["order_key"] = order_items["order_id"].map(order_key_map).astype("Int64")
    order_items["product_key"] = order_items["product_id"].map(product_key_map).astype("Int64")
    order_items["seller_key"] = order_items["seller_id"].map(seller_key_map).astype("Int64")

    fk_report_and_validate(
        order_items,
        [("order_key", "order_id"), ("product_key", "product_id"), ("seller_key", "seller_id")],
        "fact_order_items",
    )

    cols = ["order_key", "order_id", "order_item_seq", "product_key", "seller_key", "price", "freight_value"]
    bulk_copy(conn, "fact_order_items", cols, df_to_rows(order_items, cols))

    # 7) FACT_REVIEWS
    reviews_full = pd.read_parquet(PROCESSED_DIR / "reviews_full.parquet")
    reviews_full["order_key"] = reviews_full["order_id"].map(order_key_map).astype("Int64")
    reviews_full["review_date_key"] = pd.array(reviews_full["review_creation_date"].apply(to_date_key), dtype="Int64")
    reviews_full = to_nullable_int(reviews_full, ["response_days"])

    fk_report_and_validate(reviews_full, [("order_key", "order_id")], "fact_reviews")

    cols = ["review_id", "order_key", "order_id", "review_date_key", "review_creation_date",
            "review_answer_timestamp", "review_score", "review_comment_title", "review_comment_message",
            "has_logistics_keyword", "has_refund_keyword", "response_days",
            "is_most_recent_review", "review_id_reused_across_orders"]
    bulk_copy(conn, "fact_reviews", cols, df_to_rows(reviews_full, cols))

    conn.commit()
    print("\nCOMMIT — ETL succeeded, all tables loaded atomically.")
    print_final_summary(conn, ["dim_geolocation", "dim_customers", "dim_products", "dim_sellers",
                                "fact_orders", "fact_order_items", "fact_reviews"])

except Exception as e:
    conn.rollback()
    print(f"\nROLLBACK — ETL failed, warehouse left unchanged (all-or-nothing).\nReason: {e}")
    raise

finally:
    conn.close()

  loaded  19,001 rows -> dim_geolocation
  loaded  96,096 rows -> dim_customers
  loaded  32,951 rows -> dim_products
  loaded   3,095 rows -> dim_sellers

Dimensions loaded (uncommitted). Fetching surrogate keys within the same transaction...

--- fact_orders: FK validation ---
  source rows: 99,441
  unresolved customer_key           (via customer_unique_id): 0
  rows to load: 99,441   rows rejected: 0
  ✓ all foreign keys resolved
  loaded  99,441 rows -> fact_orders

--- fact_order_items: FK validation ---
  source rows: 112,650
  unresolved order_key              (via order_id): 0
  unresolved product_key            (via product_id): 0
  unresolved seller_key             (via seller_id): 0
  rows to load: 112,650   rows rejected: 0
  ✓ all foreign keys resolved
  loaded 112,650 rows -> fact_order_items

--- fact_reviews: FK validation ---
  source rows: 99,224
  unresolved order_key              (via order_id): 0
  rows to load: 99,224   rows rejected: 0
  ✓ all foreign keys resol

In [2]:
products = pd.read_parquet("../dataset/processed/products_clean.parquet")

print(products["product_category_name_english"].isna().sum())
print(products["product_category_name"].isna().sum())

610
610


In [1]:
## some checks 
import pandas as pd

items = pd.read_parquet("../dataset/processed/order_items.parquet")

print("Rows:", len(items))
print("Raw freight:", items["freight_value"].sum())

Rows: 112650
Raw freight: 2251909.54
